# 04 — Integrated Dataset / Final Merge → NEU Dataset Preparation

## Purpose

Build the canonical bridge-level ML dataset from the validated PostgreSQL layers:

```text
final.bridge
      +
final.weather
      +
final.traffic
      ↓
bridge_ml_dataset_neu.csv
```

**Final grain:** one row per bridge (`bridge_id`).

This notebook performs only the controlled integration of `final.bridge`, `final.weather`, and `final.traffic`. It preserves missing Weather/Traffic values and exports the canonical **NEU (pre-imputation)** dataset for Notebook 05.

**No imputation, PostgreSQL final-table loading, model training, tuning, or frozen-model modification is performed here.**


## 1. Required source-table gate


In [1]:
# 04 — PostgreSQL connection and canonical source configuration
from getpass import getpass
from pathlib import Path
import os
import pandas as pd
from sqlalchemy import create_engine, URL, text

# ============================================================
# PROJECT ROOT — portable across computers
# ============================================================
def find_project_root():
    env_root = os.getenv("BRIDGE_PROJECT_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / "Dataset_PlanA-B").exists():
            return root
        raise FileNotFoundError(
            f"BRIDGE_PROJECT_ROOT does not contain Dataset_PlanA-B: {root}"
        )

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Dataset_PlanA-B").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Set BRIDGE_PROJECT_ROOT to the project folder."
    )

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"
OUTPUT_DIR = OUTPUT_ROOT / "04_Integrated_Dataset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# PostgreSQL
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "Final_Project"
DB_USER = "postgres"
DB_PASSWORD = getpass("PostgreSQL password: ")

url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)
engine = create_engine(url, connect_args={"connect_timeout": 10})

# Canonical PostgreSQL source layers
BASE_TABLE = '"final"."bridge"'
WEATHER_TABLE = '"final"."weather"'
TRAFFIC_TABLE = '"final"."traffic"'

# Canonical Notebook 04 outputs
NEU_CSV_FILE = OUTPUT_DIR / "bridge_ml_dataset_neu.csv"
NEU_PARQUET_FILE = OUTPUT_DIR / "bridge_ml_dataset_neu.parquet"
DATA_INVENTORY_FILE = OUTPUT_DIR / "bridge_ml_dataset_neu_data_inventory.csv"
DATA_MANIFEST_FILE = OUTPUT_DIR / "04_data_manifest.txt"

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Sources:", BASE_TABLE, WEATHER_TABLE, TRAFFIC_TABLE)
print("NEU CSV:", NEU_CSV_FILE)
print("NEU Parquet:", NEU_PARQUET_FILE)

with engine.connect() as conn:
    print("PostgreSQL connection: PASS")
    print("Current database:", conn.execute(text("SELECT current_database()")).scalar())


Project root: C:\Datenanalyse\final Project
Dataset root: C:\Datenanalyse\final Project\Dataset_PlanA-B
Output root: C:\Datenanalyse\final Project\Output_PlanA-B
Sources: "final"."bridge" "final"."weather" "final"."traffic"
NEU CSV: C:\Datenanalyse\final Project\Output_PlanA-B\04_Integrated_Dataset\bridge_ml_dataset_neu.csv
NEU Parquet: C:\Datenanalyse\final Project\Output_PlanA-B\04_Integrated_Dataset\bridge_ml_dataset_neu.parquet
PostgreSQL connection: PASS
Current database: Final_Project


## 1A. DATA SOURCE / INPUT–OUTPUT MANIFEST

| Layer | Source / origin | Transfer method | Used by Notebook 04 | Output / destination |
|---|---|---|---|---|
| Bridge | PostgreSQL `final.bridge` | SQL query | Base population | Integrated dataset |
| Weather | PostgreSQL `final.weather` | SQL query | Bridge-year weather features | Integrated dataset |
| Traffic | PostgreSQL `final.traffic` | SQL query | Bridge-level traffic features | Integrated dataset |
| Integration | Three PostgreSQL final layers | pandas merge / aggregation | Yes | `bridge_ml_dataset_neu` |
| Canonical CSV | Notebook 04 `final_df` | Local file write | Downstream input for Notebook 05 | `Output_PlanA-B/04_Integrated_Dataset/bridge_ml_dataset_neu.csv` |
| Canonical Parquet | Notebook 04 `final_df` | Local file write | Analytical/cache copy | `Output_PlanA-B/04_Integrated_Dataset/bridge_ml_dataset_neu.parquet` |
| Data inventory | `final_df` columns | Generated from actual dataframe | Provenance / audit | `bridge_ml_dataset_neu_data_inventory.csv` |

**Important:** Notebook 04 does **not** download BASt, DWD, or Traffic source files directly. Those sources are processed upstream and arrive here through the canonical PostgreSQL tables.

### Transfer chain
```text
Notebook 01 → final.bridge   ┐
Notebook 03 → final.weather  ├→ Notebook 04 → NEU CSV/Parquet → Notebook 05
Notebook 02 → final.traffic  ┘
```


## 2. Source schemas and bridge keys


In [2]:
import pandas as pd
from sqlalchemy import inspect

# Read the REAL PostgreSQL source schemas directly.
# This avoids problems caused by quoted identifiers such as "final"."bridge".
db_inspector = inspect(engine)

def get_columns(full_table_name):
    parts = full_table_name.split(".", 1)
    if len(parts) != 2:
        raise ValueError(f"Invalid table reference: {full_table_name}")

    schema = parts[0].strip('"')
    table = parts[1].strip('"')

    columns = db_inspector.get_columns(table, schema=schema)
    return pd.DataFrame([
        {
            "ordinal_position": i + 1,
            "column_name": col["name"],
            "data_type": str(col["type"]),
        }
        for i, col in enumerate(columns)
    ])

base_columns = get_columns(BASE_TABLE)
weather_columns = get_columns(WEATHER_TABLE)
traffic_columns = get_columns(TRAFFIC_TABLE)

def require_column(columns_df, column_name, table_name):
    if column_name not in set(columns_df["column_name"]):
        available = sorted(columns_df["column_name"].tolist())
        raise RuntimeError(
            f'Required column "{column_name}" missing from {table_name}.'
            f" Available columns: {available}"
        )

# IMPORTANT: validate the source keys BEFORE the id_nr -> bridge_id normalization.
require_column(base_columns, "id_nr", BASE_TABLE)
require_column(weather_columns, "id_nr", WEATHER_TABLE)
require_column(traffic_columns, "bridge_id", TRAFFIC_TABLE)

print("Bridge key mapping:")
print("  final.bridge.id_nr       -> bridge_id")
print("  final.weather.id_nr      -> bridge_id")
print("  final.traffic.bridge_id  -> bridge_id")
print("Source-key schema gate: PASS")


Bridge key mapping:
  final.bridge.id_nr       -> bridge_id
  final.weather.id_nr      -> bridge_id
  final.traffic.bridge_id  -> bridge_id
Source-key schema gate: PASS


## 3. Load canonical final layers


In [3]:
base_df = pd.read_sql(text(f"SELECT * FROM {BASE_TABLE}"), engine)
weather_df = pd.read_sql(text(f"SELECT * FROM {WEATHER_TABLE}"), engine)
traffic_df = pd.read_sql(text(f"SELECT * FROM {TRAFFIC_TABLE}"), engine)

base_df = base_df.rename(columns={"id_nr": "bridge_id"})
weather_df = weather_df.rename(columns={"id_nr": "bridge_id"})

print("Base shape:", base_df.shape)
print("Weather shape:", weather_df.shape)
print("Traffic shape:", traffic_df.shape)


Base shape: (52214, 26)
Weather shape: (2337980, 20)
Traffic shape: (31290, 21)


## 4. Source-level uniqueness QC

Bridge and Traffic must be one row per bridge. Weather is intentionally bridge-year level.


In [4]:
def uniqueness_report(df, key, name):
    counts = df.groupby(key, dropna=False).size()
    return {
        "table": name,
        "rows": len(df),
        "distinct_bridges": df[key].nunique(dropna=True),
        "missing_key": int(df[key].isna().sum()),
        "duplicate_keys": int((counts > 1).sum()),
        "max_rows_per_key": int(counts.max()) if len(counts) else 0,
    }

source_uniqueness = pd.DataFrame([
    uniqueness_report(base_df, "bridge_id", BASE_TABLE),
    uniqueness_report(weather_df, "bridge_id", WEATHER_TABLE),
    uniqueness_report(traffic_df, "bridge_id", TRAFFIC_TABLE),
])
display(source_uniqueness)

assert base_df["bridge_id"].notna().all()
assert base_df["bridge_id"].is_unique
assert traffic_df["bridge_id"].notna().all()
assert traffic_df["bridge_id"].is_unique
assert weather_df["bridge_id"].notna().all()

print("Base and Traffic bridge-level uniqueness: PASS")


,table,rows,distinct_bridges,missing_key,duplicate_keys,max_rows_per_key
0,"""final"".""bridge""",52214,52214,0,0,1
1,"""final"".""weather""",2337980,51382,0,51119,125
2,"""final"".""traffic""",31290,31290,0,0,1


Base and Traffic bridge-level uniqueness: PASS


## 5. Collapse Weather from bridge-year to bridge-level

The annual Weather measurements are aggregated across available years using mean/min/max/std. Station and source fields are retained when constant within a bridge.


In [5]:
def collapse_weather_to_bridge_level(df):
    key = "bridge_id"

    if not df[key].duplicated().any():
        result = df.copy()
        result["weather_years_available"] = 1
        return result

    measurement_cols = [
        c for c in [
            "observation_days", "temperature_observation_days",
            "precipitation_observation_days", "temperature_mean_c",
            "temperature_min_c", "temperature_max_c",
            "precipitation_total_mm", "precipitation_max_daily_mm",
            "frost_days", "hot_days", "precipitation_days",
            "weather_distance_km"
        ] if c in df.columns
    ]

    if not measurement_cols:
        raise RuntimeError("No recognized Weather measurement columns found.")

    grouped = df.groupby(key, dropna=False)[measurement_cols].agg(["mean","min","max","std"])
    grouped.columns = [f"{c}_{stat}" for c, stat in grouped.columns]
    grouped = grouped.reset_index()

    coverage = (
        df.groupby(key, dropna=False).size()
          .rename("weather_years_available")
          .reset_index()
    )
    result = grouped.merge(coverage, on=key, how="left")

    for c in ["weather_station_id", "weather_source"]:
        if c in df.columns:
            nunique = df.groupby(key, dropna=False)[c].nunique(dropna=True)
            if nunique.max() > 1:
                raise RuntimeError(f"{c} is not constant within bridge.")
            first_values = (
                df.groupby(key, dropna=False)[c].first()
                  .rename(c).reset_index()
            )
            result = result.merge(first_values, on=key, how="left")

    return result

weather_bridge_df = collapse_weather_to_bridge_level(weather_df)

print("Weather rows before:", f"{len(weather_df):,}")
print("Weather rows after:", f"{len(weather_bridge_df):,}")
assert weather_bridge_df["bridge_id"].notna().all()
assert weather_bridge_df["bridge_id"].is_unique
print("Weather bridge-level collapse: PASS")


Weather rows before: 2,337,980
Weather rows after: 51,382
Weather bridge-level collapse: PASS


## 6. Traffic feature gate


In [6]:
required_traffic_features = [
    "dtv_latest","dtv_mean","dtv_max","dtv_min","dtv_std",
    "heavy_vehicle_mean","heavy_vehicle_max","heavy_vehicle_share_mean",
    "dtv_yoy_growth_mean","dtv_yoy_growth_max","dtv_yoy_growth_min",
    "dtv_trend_per_year","traffic_data_coverage"
]

missing_traffic = [c for c in required_traffic_features if c not in traffic_df.columns]
if missing_traffic:
    raise RuntimeError(f"Missing required Traffic features: {missing_traffic}")

share = traffic_df["heavy_vehicle_share_mean"]
invalid_share = int(((share < 0) | (share > 1)).fillna(False).sum())

print("Invalid heavy_vehicle_share_mean:", invalid_share)
assert invalid_share == 0
print("Traffic feature and heavy-vehicle-share gate: PASS")


Invalid heavy_vehicle_share_mean: 0
Traffic feature and heavy-vehicle-share gate: PASS


## 7. Prefix Weather and Traffic columns and merge

The Bridge layer remains the base population. Weather and Traffic are left-joined. Missing enrichment values are preserved.


In [7]:
def prefix_non_key_columns(df, prefix, key="bridge_id"):
    return df.rename(columns={c: prefix + c for c in df.columns if c != key})

weather_for_merge = prefix_non_key_columns(weather_bridge_df, "weather_")
traffic_for_merge = prefix_non_key_columns(traffic_df, "traffic_")

# bridge_id is the intentional join key and must not be treated as a collision.
base_non_key = set(base_df.columns) - {"bridge_id"}
weather_non_key = set(weather_for_merge.columns) - {"bridge_id"}
traffic_non_key = set(traffic_for_merge.columns) - {"bridge_id"}

overlap = (
    (base_non_key & weather_non_key)
    | (base_non_key & traffic_non_key)
    | (weather_non_key & traffic_non_key)
)
if overlap:
    raise RuntimeError(f"Column collision detected: {sorted(overlap)}")

final_df = (
    base_df
    .merge(weather_for_merge, on="bridge_id", how="left", validate="one_to_one")
    .merge(traffic_for_merge, on="bridge_id", how="left", validate="one_to_one")
)

print("Final shape:", final_df.shape)
print("Distinct bridges:", f"{final_df['bridge_id'].nunique():,}")
print("Duplicate bridge IDs:", int(final_df["bridge_id"].duplicated().sum()))

assert len(final_df) == len(base_df)
assert final_df["bridge_id"].notna().all()
assert final_df["bridge_id"].is_unique
assert final_df.columns.is_unique

print("Controlled integration: PASS")


Final shape: (52214, 97)
Distinct bridges: 52,214
Duplicate bridge IDs: 0
Controlled integration: PASS


## 8. Coverage, feature inventory and missingness


In [8]:
coverage = pd.DataFrame([{
    "bridges_base": base_df["bridge_id"].nunique(),
    "bridges_final": final_df["bridge_id"].nunique(),
    "bridges_with_weather": int(final_df["weather_weather_station_id"].notna().sum())
        if "weather_weather_station_id" in final_df.columns else 0,
    "bridges_with_traffic": int(final_df["traffic_traffic_station_id"].notna().sum())
        if "traffic_traffic_station_id" in final_df.columns else 0,
    "rows_final": len(final_df),
    "columns_final": len(final_df.columns),
}])
display(coverage)

inventory = pd.DataFrame({"column": final_df.columns})
base_names = set(base_df.columns)

def source_of(column):
    if column in base_names:
        return "bridge_base"
    if column.startswith("weather_"):
        return "weather"
    if column.startswith("traffic_"):
        return "traffic"
    return "other"

inventory["source"] = inventory["column"].map(source_of)
inventory["dtype"] = [str(final_df[c].dtype) for c in final_df.columns]
inventory["missing_pct"] = [round(final_df[c].isna().mean()*100, 2) for c in final_df.columns]

display(inventory.sort_values(["source","column"]).reset_index(drop=True))
display(inventory["source"].value_counts().to_frame("feature_count"))

missingness = (
    final_df.isna().mean().mul(100).round(2)
    .sort_values(ascending=False).rename("missing_pct").to_frame()
)
display(missingness.head(30))

assert coverage.loc[0, "bridges_base"] == coverage.loc[0, "bridges_final"]
print("Bridge population preservation: PASS")


,bridges_base,bridges_final,bridges_with_weather,bridges_with_traffic,rows_final,columns_final
0,52214,52214,51382,31290,52214,97


,column,source,dtype,missing_pct
0,altersklasse,bridge_base,str,0.00
1,baujahr,bridge_base,int64,0.00
2,baustoffklasse,bridge_base,str,0.00
3,bauwerk,bridge_base,str,0.00
4,bauwerksart_text,bridge_base,str,0.00
...,...,...,...,...
92,weather_weather_distance_km_min,weather,float64,1.59
93,weather_weather_distance_km_std,weather,float64,2.10
94,weather_weather_source,weather,str,1.59
95,weather_weather_station_id,weather,str,1.59


,feature_count
source,
weather,51
bridge_base,26
traffic,20


,missing_pct
traffic_dtv_yoy_growth_mean,40.93
traffic_dtv_yoy_growth_max,40.93
traffic_dtv_yoy_growth_min,40.93
traffic_dtv_std,40.77
traffic_dtv_trend_per_year,40.77
traffic_traffic_last_year,40.33
traffic_heavy_vehicle_share_mean,40.33
traffic_dtv_latest,40.33
traffic_heavy_vehicle_mean,40.33
traffic_dtv_max,40.33


Bridge population preservation: PASS


## 9. Final ML-readiness structural QC


In [9]:
assert final_df["bridge_id"].notna().all()
assert final_df["bridge_id"].is_unique
assert final_df.columns.is_unique

target_like = {"target","target_case_a","target_case_b","case_a","case_b"}
found_targets = [c for c in final_df.columns if c.lower() in target_like]
if found_targets:
    raise RuntimeError(f"Unexpected target-like columns found: {found_targets}")

if "zustandsnote" in final_df.columns:
    condition = pd.to_numeric(final_df["zustandsnote"], errors="coerce")
    invalid_condition = int(((condition < 1) | (condition > 4)).fillna(False).sum())
    print("Invalid Zustandsnote values:", invalid_condition)
    assert invalid_condition == 0

if "baujahr" in final_df.columns:
    year = pd.to_numeric(final_df["baujahr"], errors="coerce")
    assert year.notna().all()
    assert (year > 1900).all()

print("Final structural / ML-readiness gate: PASS")


Invalid Zustandsnote values: 0
Final structural / ML-readiness gate: PASS


## 10. Prepare the canonical NEU dataset

The integrated DataFrame is now validated and prepared for Notebook 05. This stage does **not** create the final PostgreSQL ML table.


In [10]:
# PostgreSQL final-table creation is intentionally not performed in Notebook 04.
assert final_df["bridge_id"].notna().all()
assert final_df["bridge_id"].is_unique
print("NEU dataset preparation gate: PASS")
print("PostgreSQL final-table creation: deferred to Notebook 06")


NEU dataset preparation gate: PASS
PostgreSQL final-table creation: deferred to Notebook 06


## 11. Export canonical NEU CSV and Parquet snapshots


The validated bridge-level DataFrame is exported here as the canonical **pre-imputation NEU dataset**.

These files are the only dataset inputs required by Notebook 05:
- `bridge_ml_dataset_neu.csv`
- `bridge_ml_dataset_neu.parquet`

No PostgreSQL final ML table is created in this notebook.


## 12. Final DataFrame / file validation

The integrated dataset is validated against the exported canonical NEU CSV and Parquet snapshots.

**PostgreSQL final-table creation is intentionally deferred to Notebook 06.**


In [11]:
# Export canonical NEU snapshots
final_df.to_csv(NEU_CSV_FILE, index=False)
final_df.to_parquet(NEU_PARQUET_FILE, index=False)

print("CSV exported:", NEU_CSV_FILE)
print("Parquet exported:", NEU_PARQUET_FILE)

# Reload exported files and validate preservation
csv_check = pd.read_csv(NEU_CSV_FILE)
parquet_check = pd.read_parquet(NEU_PARQUET_FILE)

validation = pd.DataFrame([{
    "dataframe_rows": len(final_df),
    "csv_rows": len(csv_check),
    "parquet_rows": len(parquet_check),
    "dataframe_bridges": final_df["bridge_id"].nunique(),
    "csv_bridges": csv_check["bridge_id"].nunique(),
    "parquet_bridges": parquet_check["bridge_id"].nunique(),
    "csv_columns": len(csv_check.columns),
    "parquet_columns": len(parquet_check.columns),
}])
display(validation)

assert len(final_df) == len(csv_check)
assert len(final_df) == len(parquet_check)
assert final_df["bridge_id"].nunique() == csv_check["bridge_id"].nunique()
assert final_df["bridge_id"].nunique() == parquet_check["bridge_id"].nunique()
assert csv_check["bridge_id"].is_unique
assert parquet_check["bridge_id"].is_unique
assert len(csv_check.columns) == len(final_df.columns)
assert len(parquet_check.columns) == len(final_df.columns)

print("Export preservation validation: PASS")


CSV exported: C:\Datenanalyse\final Project\Output_PlanA-B\04_Integrated_Dataset\bridge_ml_dataset_neu.csv
Parquet exported: C:\Datenanalyse\final Project\Output_PlanA-B\04_Integrated_Dataset\bridge_ml_dataset_neu.parquet


,dataframe_rows,csv_rows,parquet_rows,dataframe_bridges,csv_bridges,parquet_bridges,csv_columns,parquet_columns
0,52214,52214,52214,52214,52214,52214,97,97


Export preservation validation: PASS


In [12]:
# ============================================================
# DATA INVENTORY — actual columns in the canonical NEU dataset
# ============================================================
base_names = set(base_df.columns)

def source_of(column):
    if column in base_names:
        return "bridge_base"
    if column.startswith("weather_"):
        return "weather"
    if column.startswith("traffic_"):
        return "traffic"
    return "derived_integration"

data_inventory = pd.DataFrame({
    "column_order": range(1, len(final_df.columns) + 1),
    "column": final_df.columns,
    "source_layer": [source_of(c) for c in final_df.columns],
    "dtype": [str(final_df[c].dtype) for c in final_df.columns],
    "missing_count": [int(final_df[c].isna().sum()) for c in final_df.columns],
    "missing_pct": [round(float(final_df[c].isna().mean() * 100), 2) for c in final_df.columns],
})

data_inventory.to_csv(DATA_INVENTORY_FILE, index=False)
display(data_inventory)
print("Data inventory exported:", DATA_INVENTORY_FILE)


,column_order,column,source_layer,dtype,missing_count,missing_pct
0,1,bridge_id,bridge_base,str,0,0.00
1,2,bauwerk,bridge_base,str,0,0.00
2,3,bauwerksart_text,bridge_base,str,0,0.00
3,4,stadium_text,bridge_base,str,0,0.00
4,5,bwnr,bridge_base,int64,0,0.00
...,...,...,...,...,...,...
92,93,traffic_dtv_yoy_growth_mean,traffic,float64,21372,40.93
93,94,traffic_dtv_yoy_growth_max,traffic,float64,21372,40.93
94,95,traffic_dtv_yoy_growth_min,traffic,float64,21372,40.93
95,96,traffic_dtv_trend_per_year,traffic,float64,21290,40.77


Data inventory exported: C:\Datenanalyse\final Project\Output_PlanA-B\04_Integrated_Dataset\bridge_ml_dataset_neu_data_inventory.csv


In [13]:
# ============================================================
# SAVE HUMAN-READABLE DATA MANIFEST
# ============================================================
manifest_text = f"""Notebook 04 — Integrated Dataset Data Manifest
=================================================

PROJECT_ROOT: {PROJECT_ROOT}
DATASET_ROOT: {DATASET_ROOT}
OUTPUT_DIR: {OUTPUT_DIR}

INPUTS
------
PostgreSQL database: {DB_NAME}
  - {BASE_TABLE}
  - {WEATHER_TABLE}
  - {TRAFFIC_TABLE}

PROCESSING
----------
Bridge is the base population.
Weather is collapsed from bridge-year to bridge-level.
Traffic is joined at bridge level.
Missing Weather/Traffic values are preserved.
Final grain: one row per bridge_id.

OUTPUTS
-------
CSV:       {NEU_CSV_FILE}
Parquet:   {NEU_PARQUET_FILE}
Inventory: {DATA_INVENTORY_FILE}

DOWNSTREAM
----------
Notebook 05 consumes the canonical NEU dataset.
No imputation or ML training is performed in Notebook 04.
"""

DATA_MANIFEST_FILE.write_text(manifest_text, encoding="utf-8")
print("Data manifest exported:", DATA_MANIFEST_FILE)


Data manifest exported: C:\Datenanalyse\final Project\Output_PlanA-B\04_Integrated_Dataset\04_data_manifest.txt


## Final validation gate

The integrated dataset is accepted only if the canonical source tables exist, the bridge population is preserved, `bridge_id` is unique, no duplicate bridge rows are introduced, `zustandsnote` is present, the bridge-level feature families are present, and missing values are preserved.

**PostgreSQL final-table creation is intentionally deferred to Notebook 06.**


In [14]:
# Final structural gate
assert "bridge_id" in final_df.columns
assert "zustandsnote" in final_df.columns
assert final_df["bridge_id"].notna().all()
assert final_df["bridge_id"].is_unique
assert len(final_df) == len(base_df)
assert final_df["bridge_id"].nunique() == base_df["bridge_id"].nunique()

print("FINAL INTEGRATED DATASET")
print("------------------------")
print("Rows:", f"{len(final_df):,}")
print("Columns:", f"{len(final_df.columns):,}")
print("Unique bridges:", f"{final_df['bridge_id'].nunique():,}")
print("Duplicate bridge IDs:", int(final_df["bridge_id"].duplicated().sum()))
print("Target present: PASS")
print("Bridge population preserved: PASS")
print("One row per bridge: PASS")


FINAL INTEGRATED DATASET
------------------------
Rows: 52,214
Columns: 97
Unique bridges: 52,214
Duplicate bridge IDs: 0
Target present: PASS
Bridge population preserved: PASS
One row per bridge: PASS


## Final output

Canonical **pre-imputation NEU dataset** for Notebook 05:

- `bridge_ml_dataset_neu.csv`
- `bridge_ml_dataset_neu.parquet`

**No PostgreSQL final ML table is created here.** Notebook 06 is the only notebook responsible for loading the imputed dataset into `final.bridge_ml_dataset_final`.

**Status:** READY for Notebook 05 — Dataset Imputation.
